# embedding EDA

In this notebook, we figure out the huggingface embedding dataset.
We need to map these to the page ids.
We also need to get one embedding per page.
Then we can also explore what it'll take in order to index this into faiss (with a 1-10% sample).

In [2]:
from pathlib import Path
from pyspark.sql import SparkSession


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .config("spark.jars.packages", "graphframes:graphframes:0.8.4-spark3.5-s_2.13")
        .getOrCreate()
    )
    return spark


root = Path("~/scratch/trec-tot-2025/data").expanduser()
knn_root = root / "enwiki/processed/graph/v2/bge-m3-knn"

spark = get_spark()
nodes = spark.read.parquet(str(knn_root / "nodes"))
edges = spark.read.parquet(str(knn_root / "edges"))
nodes.printSchema()
edges.printSchema()
(nodes.count(), edges.count())

root
 |-- id: long (nullable = true)

root
 |-- src: long (nullable = true)
 |-- dst: long (nullable = true)
 |-- score: float (nullable = true)
 |-- rank: integer (nullable = true)



(6614232, 324097368)

In [3]:
from graphframes import GraphFrame

g = GraphFrame(nodes, edges)
# check for in-degree
in_degree = g.inDegrees
in_degree.show()

+--------+--------+
|      id|inDegree|
+--------+--------+
|65691009|      79|
| 1086266|     206|
|62459218|     114|
|55819382|      76|
| 7419941|     112|
|63666072|      42|
| 5909866|      75|
| 1925182|      69|
| 1268949|      22|
| 7819909|     106|
| 7146282|      28|
|22605964|      40|
| 7154969|     130|
| 1870746|      88|
| 9767890|     104|
|31094668|      47|
| 3620930|     185|
|   32571|     691|
| 2149752|      39|
|34888068|      96|
+--------+--------+
only showing top 20 rows


In [4]:
in_degree.describe().show()

+-------+--------------------+------------------+
|summary|                  id|          inDegree|
+-------+--------------------+------------------+
|  count|             6599864|           6599864|
|   mean| 3.312341742506194E7|49.106673713276514|
| stddev|2.3064207013063814E7| 43.35120725332105|
|    min|                  12|                 1|
|    max|            77104121|              1847|
+-------+--------------------+------------------+



In [5]:
g.outDegrees.describe().show()

+-------+--------------------+---------+
|summary|                  id|outDegree|
+-------+--------------------+---------+
|  count|             6614232|  6614232|
|   mean|3.3128307348651364E7|     49.0|
| stddev|2.3063246322609715E7|      0.0|
|    min|                  12|       49|
|    max|            77104121|       49|
+-------+--------------------+---------+

